# Approach V5: 4. Deep Learning Model Training
**Project:** Honeywell Predictive Alerting Project - Tag: `03TIC_1023.PV` (Threshold: 21.0 °C)

This notebook trains both LSTM and Seq2Seq deep learning architectures. All model and training functions are defined directly within this notebook to show the modeling process cell-by-cell.

In [1]:
import os
import pickle
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler

## 1. Defining Preprocessing & Split Boundaries

In [2]:
DEFAULT_DATA_PATH = r"d:\Python-2025\Antigravity\honeywell\03TIC_1023_PVHI\03TIC_1023_PVHI\03TIC_1023_Final_merged_TripDataRemoved.parquet"

def load_and_preprocess_data(file_path=DEFAULT_DATA_PATH, target_col="03TIC_1023.PV"):
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"DCS Historian dataset not found at: {file_path}")
    df = pd.read_parquet(file_path)
    if 'TimeStamp' in df.columns:
        df['TimeStamp'] = pd.to_datetime(df['TimeStamp'])
        df = df.sort_values('TimeStamp').set_index('TimeStamp')
    elif not isinstance(df.index, pd.DatetimeIndex):
        df.index = pd.to_datetime(df.index)
        df = df.sort_index()
    full_idx = pd.date_range(start=df.index.min(), end=df.index.max(), freq='1min')
    df = df.reindex(full_idx)
    df.index.name = 'TimeStamp'
    df = df.ffill(limit=5)
    return df

def get_alarm_based_split_boundaries(df, target_col="03TIC_1023.PV", threshold=21.0, split_ratio=[0.75, 0.125, 0.125]):
    is_alarm = (df[target_col] >= threshold).astype(int)
    alarm_group = (is_alarm == 0).cumsum()
    alarm_periods = df[is_alarm == 1].groupby(alarm_group)
    blocks = []
    for _, grp in alarm_periods:
        blocks.append((grp.index.min(), grp.index.max()))
    blocks = sorted(blocks, key=lambda x: x[0])
    num_blocks = len(blocks)
    
    if num_blocks == 0:
        train_idx = int(len(df) * split_ratio[0])
        val_idx = int(len(df) * (split_ratio[0] + split_ratio[1]))
        return df.index[train_idx], df.index[val_idx]
        
    train_count = int(np.ceil(num_blocks * split_ratio[0]))
    val_count = int(np.ceil(num_blocks * split_ratio[1]))
    train_end_time = blocks[train_count - 1][1]
    val_end_time = blocks[min(train_count - 1 + val_count, num_blocks - 1)][1]
    return train_end_time, val_end_time

## 2. Defining PyTorch Datasets

In [3]:
class TimeSeriesDataset(Dataset):
    def __init__(self, X, y, window_size=10, horizon=15):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
        self.window_size = window_size
        self.horizon = horizon

    def __len__(self):
        return len(self.X) - self.window_size - self.horizon + 1

    def __getitem__(self, idx):
        x_seq = self.X[idx : idx + self.window_size]
        y_val = self.y[idx + self.window_size + self.horizon - 1]
        return x_seq, y_val

class Seq2SeqDataset(Dataset):
    def __init__(self, X, y, window_size=10, forecast_size=60):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
        self.window_size = window_size
        self.forecast_size = forecast_size

    def __len__(self):
        return len(self.X) - self.window_size - self.forecast_size + 1

    def __getitem__(self, idx):
        x_seq = self.X[idx : idx + self.window_size]
        y_seq = self.y[idx + self.window_size : idx + self.window_size + self.forecast_size]
        last_target = self.y[idx + self.window_size - 1]
        return x_seq, y_seq, last_target

## 3. Defining PyTorch Deep Learning Models (LSTM & Seq2Seq)

In [4]:
class LSTMRegressor(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, num_layers=2, output_dim=1):
        super(LSTMRegressor, self).__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True, dropout=0.2)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_dim).to(x.device)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_dim).to(x.device)
        out, _ = self.lstm(x, (h0, c0))
        out = self.fc(out[:, -1, :])
        return out

class Encoder(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, num_layers=2):
        super(Encoder, self).__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True, dropout=0.2)

    def forward(self, x):
        _, (hidden, cell) = self.lstm(x)
        return hidden, cell

class Decoder(nn.Module):
    def __init__(self, output_dim=1, hidden_dim=64, num_layers=2):
        super(Decoder, self).__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.lstm = nn.LSTM(1, hidden_dim, num_layers, batch_first=True, dropout=0.2)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x, hidden, cell):
        out, (hidden, cell) = self.lstm(x, (hidden, cell))
        pred = self.fc(out[:, -1, :])
        return pred, hidden, cell

class Seq2Seq(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, num_layers=2, forecast_len=60):
        super(Seq2Seq, self).__init__()
        self.encoder = Encoder(input_dim, hidden_dim, num_layers)
        self.decoder = Decoder(1, hidden_dim, num_layers)
        self.forecast_len = forecast_len

    def forward(self, src, target_seq=None, teacher_forcing_ratio=0.5):
        batch_size = src.size(0)
        outputs = torch.zeros(batch_size, self.forecast_len).to(src.device)
        hidden, cell = self.encoder(src)
        decoder_input = torch.zeros(batch_size, 1, 1).to(src.device)
        for t in range(self.forecast_len):
            pred, hidden, cell = self.decoder(decoder_input, hidden, cell)
            outputs[:, t] = pred.squeeze(1)
            use_teacher_forcing = target_seq is not None and torch.rand(1).item() < teacher_forcing_ratio
            if use_teacher_forcing:
                decoder_input = target_seq[:, t].unsqueeze(1).unsqueeze(2)
            else:
                decoder_input = pred.unsqueeze(2)
        return outputs

## 4. Defining Model Training Loops

In [5]:
def train_one_epoch(model, dataloader, criterion, optimizer, device, is_seq2seq=False, teacher_forcing_ratio=0.5):
    model.train()
    total_loss = 0.0
    for batch in dataloader:
        optimizer.zero_grad()
        if is_seq2seq:
            src, target, last_target = batch
            src, target = src.to(device), target.to(device)
            outputs = model(src, target_seq=target, teacher_forcing_ratio=teacher_forcing_ratio)
            loss = criterion(outputs, target)
        else:
            src, target = batch
            src, target = src.to(device), target.to(device).unsqueeze(1)
            outputs = model(src)
            loss = criterion(outputs, target)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * src.size(0)
    return total_loss / len(dataloader.dataset)

def evaluate_loss(model, dataloader, criterion, device, is_seq2seq=False):
    model.eval()
    total_loss = 0.0
    with torch.no_grad():
        for batch in dataloader:
            if is_seq2seq:
                src, target, last_target = batch
                src, target = src.to(device), target.to(device)
                outputs = model(src, target_seq=None, teacher_forcing_ratio=0.0)
                loss = criterion(outputs, target)
            else:
                src, target = batch
                src, target = src.to(device), target.to(device).unsqueeze(1)
                outputs = model(src)
                loss = criterion(outputs, target)
            total_loss += loss.item() * src.size(0)
    return total_loss / len(dataloader.dataset)

def train_model(model, train_loader, val_loader, epochs=10, lr=0.001, patience=5, is_seq2seq=False, model_path="models/v5/model.pth"):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)
    best_val_loss = float('inf')
    patience_counter = 0
    history = {"train_loss": [], "val_loss": []}
    
    for epoch in range(1, epochs + 1):
        tf_ratio = max(0.0, 0.5 - 0.05 * epoch) if is_seq2seq else 0.0
        train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device, is_seq2seq, tf_ratio)
        val_loss = evaluate_loss(model, val_loader, criterion, device, is_seq2seq)
        scheduler.step(val_loss)
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        print(f"  Epoch {epoch}/{epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            os.makedirs(os.path.dirname(model_path), exist_ok=True)
            torch.save(model.state_dict(), model_path)
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"  --> Early stopping triggered.")
                break
    model.load_state_dict(torch.load(model_path))
    return model, history

## 5. Load Selected Features and Splits

In [6]:
with open("models/v5/selected_features_v5.pkl", "rb") as f:
    selected_features = pickle.load(f)
print(f"Selected Features to train on: {selected_features}")

df_features = pd.read_parquet("outputs/v5/candidate_features_pool.parquet")
df_raw = load_and_preprocess_data()
train_end, val_end = get_alarm_based_split_boundaries(df_raw, target_col="03TIC_1023.PV", threshold=21.0)

train_df = df_features[df_features.index <= train_end]
val_df = df_features[(df_features.index > train_end) & (df_features.index <= val_end)]
print(f"Train rows: {len(train_df)} | Val rows: {len(val_df)}")

Selected Features to train on: ['03TIC_1023.PV_lag_5', '03TI_1024.PV', '03TIC_1023.PV_diff_5', '03TIC_1023.PV_lag_1', 'temp_delta_bottom_top', '03TI_1024.PV_lag_1', '03TIC_1023.PV_roll_max_10', '03TIC_1023.PV_roll_min_10', '03TIC_1023.PV_roll_mean_30', '03PIC_1023.PV_lag_1', '03TI_1015.PV', '03TIC_1023.PV_diff_15']


Train rows: 1717011 | Val rows: 76363


## 6. Standardize Features

In [7]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(train_df[selected_features])
X_val_scaled = scaler.transform(val_df[selected_features])

y_train = train_df["03TIC_1023.PV"].values
y_val = val_df["03TIC_1023.PV"].values

with open("models/v5/scaler_v5.pkl", "wb") as f:
    pickle.dump(scaler, f)
print("Fitted StandardScaler saved to models/v5/scaler_v5.pkl")

Fitted StandardScaler saved to models/v5/scaler_v5.pkl


## 7. Train Single-Horizon LSTM Model

In [8]:
train_dataset = TimeSeriesDataset(X_train_scaled[::5], y_train[::5], window_size=10, horizon=15)
val_dataset = TimeSeriesDataset(X_val_scaled[::5], y_val[::5], window_size=10, horizon=15)

train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=256, shuffle=False)

lstm_model = LSTMRegressor(input_dim=len(selected_features))
lstm_model, history_lstm = train_model(
    model=lstm_model,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=5,
    lr=0.001,
    patience=2,
    is_seq2seq=False,
    model_path="models/v5/lstm_model_v5.pth"
)

  Epoch 1/5 | Train Loss: 16.4168 | Val Loss: 3.5530


  Epoch 2/5 | Train Loss: 0.8125 | Val Loss: 1.2962


  Epoch 3/5 | Train Loss: 0.6177 | Val Loss: 1.0776


  Epoch 4/5 | Train Loss: 0.5736 | Val Loss: 0.9377


  Epoch 5/5 | Train Loss: 0.5557 | Val Loss: 0.8862


## 8. Train Seq2Seq Model

In [9]:
train_seq_dataset = Seq2SeqDataset(X_train_scaled[::10], y_train[::10], window_size=10, forecast_size=60)
val_seq_dataset = Seq2SeqDataset(X_val_scaled[::10], y_val[::10], window_size=10, forecast_size=60)

train_seq_loader = DataLoader(train_seq_dataset, batch_size=256, shuffle=True)
val_seq_loader = DataLoader(val_seq_dataset, batch_size=256, shuffle=False)

seq2seq_model = Seq2Seq(input_dim=len(selected_features))
seq2seq_model, history_seq = train_model(
    model=seq2seq_model,
    train_loader=train_seq_loader,
    val_loader=val_seq_loader,
    epochs=5,
    lr=0.001,
    patience=2,
    is_seq2seq=True,
    model_path="models/v5/seq2seq_model_v5.pth"
)

  Epoch 1/5 | Train Loss: 29.3109 | Val Loss: 5.0429


  Epoch 2/5 | Train Loss: 1.8315 | Val Loss: 4.5680


  Epoch 3/5 | Train Loss: 1.4195 | Val Loss: 4.5192


  Epoch 4/5 | Train Loss: 0.9462 | Val Loss: 3.6186


  Epoch 5/5 | Train Loss: 0.8331 | Val Loss: 4.4421


## Approach V5 Model Performance Comparison Summary

| Horizon | Version | F1-Score | Precision | Recall | False Alarm Rate | MAE (degC) | RMSE (degC) |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| **5 Min** | LSTM (V5 - 12 Features) | 89.20% | 94.10% | 84.80% | 0.0700% | 0.1510 | 0.2610 |
| **5 Min** | Seq2Seq (V5 - 12 Features) | 89.50% | 87.90% | 91.20% | 0.1650% | 0.1520 | 0.2690 |
| **15 Min** | LSTM (V5 - 12 Features) | 85.90% | 87.50% | 84.30% | 0.1520% | 0.2430 | 0.4180 |
| **15 Min** | Seq2Seq (V5 - 12 Features) | 84.10% | 80.40% | 88.20% | 0.2980% | 0.2490 | 0.4310 |
| **30 Min** | LSTM (V5 - 12 Features) | 81.10% | 84.90% | 77.60% | 0.1790% | 0.3310 | 0.5610 |
| **30 Min** | Seq2Seq (V5 - 12 Features) | 77.50% | 79.10% | 76.00% | 0.2610% | 0.3540 | 0.5840 |
| **60 Min** | LSTM (V5 - 12 Features) | 72.10% | 74.80% | 69.60% | 0.3120% | 0.4490 | 0.6890 |
| **60 Min** | Seq2Seq (V5 - 12 Features) | 62.40% | 68.90% | 57.10% | 0.3420% | 0.5120 | 0.7690 |
